# 下跌延续与修复：固定两特征信息实验
从已复用区间诊断提出机制，新增距20日最低收盘的时间及5日广度变化在下跌状态中的交互。与64项旧信息及相同状态/缺失控制配对，固定6拟合，无参数搜索。所有历史均不称新鲜样本外。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from src.model.revenue_forecast import RevenueForecast
from src.workflow.fed_capacity_information import evaluation_mask,_bootstrap
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/drawdown_transition_information_20261014';plan=json.loads((card/'plan.json').read_text())
qlib.init(provider_uri=str(root/'data/qlib/full_path_tiered_allocation_20261013'),region='us')
def read(source,name):
 p=root/'research'/source;s=json.loads((p/'summary.json').read_text());m=p/'verification_manifest.json';assert hashlib.sha256(m.read_bytes()).hexdigest()==plan['sources'][source]['manifest_sha256']
 e=next(e for e in json.loads(m.read_text())['entries'] if e['path'].endswith('/artifacts/'+name))
 with R.uri_context(str(p/'qlib_recorder')):
  rec=R.get_recorder(recorder_id=plan['sources'][source]['recorder_id'],experiment_name=source);a=Path(rec.download_artifact(name));assert hashlib.sha256(a.read_bytes()).hexdigest()==e['sha256'];return rec.load_object(name)
frame=read('full_path_tiered_allocation_20261013','inputs/frame');grouped=read('guidance_information_20261014','inputs/grouped')
assert pd.DatetimeIndex(grouped.index.get_level_values('datetime')).equals(frame.index)
prior=frame.close_soxx.shift(1)
age=prior.rolling(20,min_periods=20).apply(lambda a:np.argmin(a[::-1])/19.,raw=True)
breadth_change=frame.tech_breadth_20.diff(5)
decline=frame.soxx_ret5.lt(0)&frame.soxx_ret20.lt(0)
known=age.notna()&breadth_change.notna()
extra=pd.DataFrame({'decline_state':decline.astype(float),'transition_known':known.astype(float),'decline_low_age20':age.where(decline,0.),'decline_breadth_change5':breadth_change.where(decline,0.)},index=frame.index)
assert (pd.to_datetime(frame.feature_available_at)<pd.to_datetime(frame.decision_at)).all()
for c in extra:grouped[('feature',c)]=extra[c].to_numpy()
handler=DataHandlerLP.from_df(grouped);tracking=card/'qlib_recorder';audits=[];increments=[];losses={};predictions={}
with R.start(experiment_name=card.name,recorder_name='fixed_drawdown_transition',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(maximum_model_fits=6,new_backtests=0)
 rec.save_objects(artifact_path='inputs',plan=plan,grouped=grouped,transition_features=extra)
 for fold in plan['folds']:
  bounds=frame.index[frame.signal_date.between(fold['start'],fold['end'])]
  ds=DatasetH(handler=handler,segments={'train':(None,pd.Timestamp(fold['cutoff'])-pd.Timedelta(nanoseconds=1)),'test':(bounds[0],bounds[-1])})
  truth=ds.prepare('test',col_set='label')[plan['targets']];meta=ds.prepare('test',col_set='meta');valid=evaluation_mask(truth,meta,fold,plan['as_of']);pair={};scalers=[]
  for family,columns in [('control',plan['base_features']+plan['controls']),('transition',plan['base_features']+plan['controls']+plan['candidate_features'])]:
   assert len(audits)<6
   model=RevenueForecast(columns,plan['targets'],fold['cutoff'],alpha=10.).fit(ds);prediction=model.predict(ds);scalers.append(model.y_scaler.scale_)
   pair[family]=((prediction.loc[valid]-truth.loc[valid]).to_numpy()/model.y_scaler.scale_)**2
   audits.append({'fold':fold['name'],'family':family,**model.audit});predictions[(fold['name'],family)]={'prediction':prediction,'truth':truth,'valid':valid}
   rec.save_objects(artifact_path='models',**{fold['name']+'_'+family:model})
  np.testing.assert_allclose(*scalers)
  diff=pair['control']-pair['transition'];state=grouped.loc[truth.index[valid],('feature','decline_state')].to_numpy()
  # Zero contribution on other states preserves all calendar positions in the bootstrap.
  values={'joint_all':diff.mean(axis=1),'cash_decline_contribution':diff[:,0]*state}
  losses[fold['name']]=values
  for metric,value in values.items():increments.append({'fold':fold['name'],'metric':metric,'decline_sessions':int(state.sum()),**_bootstrap(value,plan)})
  print('completed',fold['name'],flush=True)
 for metric in ['joint_all','cash_decline_contribution']:
  value=np.concatenate([losses[f['name']][metric] for f in plan['folds'] if f['name'].startswith('development')]);increments.append({'fold':'pooled_development','metric':metric,**_bootstrap(value,plan)})
 table=pd.DataFrame(increments)
 passed=all(row['mean_joint_normalized_loss_improvement']>0 for row in increments if row['fold'].startswith('development')) and table.loc[table.fold.eq('pooled_development'),'lower_bound'].gt(0).all()
 summary={'experiment':card.name,'decision':'information_pass_needs_independent_confirmation' if passed else 'not_promoted_fixed_drawdown_transition_failed','information_gate_pass':bool(passed),'objective_achieved':False,'new_fits':len(audits),'new_backtests':0,'hypothesis':'decline-conditioned low-age and breadth-change distinguish continuation from repair beyond 64 existing features','selection_warning':'Hypothesis motivated by reused 2023+ failure diagnosis; 2021/2022 also previously studied. No fresh OOS claim.','provider_uri':str(root/'data/qlib/full_path_tiered_allocation_20261013'),'region':'us','as_of':plan['as_of'],'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(tracking),'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/recorder.html','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','models/*','analysis/predictions','analysis/losses']}}
 rec.save_objects(artifact_path='analysis',summary=summary,increments=table,training=audits,predictions=predictions,losses=losses)
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(completed_fits=len(audits),registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json')
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');table.to_csv(card/'information_increment.csv',index=False)
pd.DataFrame([{k:a[k] for k in ['fold','family','training_rows','latest_label_available_at','cutoff']} for a in audits]).to_csv(card/'training_audit.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':summary['provider_uri'],'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False))
print(table.to_string(index=False))


# 下跌状态修复交互：固定信息实验未通过

由差区间诊断提出固定机制：使用此前20个已完成收盘中距最近最低点的时间，以及前一信号日科技广度相对5日前的变化，分别与过去5/20日均下跌状态相乘。最低点时间只用close.shift(1)，不读当日未来收盘。

控制组64项旧信息加相同下跌/可用标志，候选仅加上述两项；相同训练数据、成熟标签和Ridge alpha=10。2021、2022、2023+共6次Qlib Dataset配对拟合，0回测；未按结果调整周期或阈值。两组训练行数依次553/805/1056，最后成熟标签均早于各自cutoff。

归一化预测损失改善（正数为好，不是收益）：2021全样本+0.00446，2022 +0.03193；合并开发+0.01817，95%区块区间[-0.10469,0.14661]。下跌状态现金损失的全日历贡献改善合并为+0.02292，区间[-0.08052,0.13081]。两个主门均未通过。

2023+复用诊断全样本-0.01168，区间[-0.02763,0.00624]；下跌状态现金贡献-0.02040，区间[-0.03610,-0.00237]，方向更差。统计包含491个开发成熟日、940个复用成熟日；对应下跌状态42/109/204日。保持其他日期贡献为零来保留区块时序，未把它当成独立下跌事件数。

决定：not_promoted_fixed_drawdown_transition_failed。两个特征没有给出稳定信息增量，固定路线关闭；不变周期、换模型或改符号继续搜。低点时间和宽泛科技ETF广度不足以证明能区分跌势延续与修复，也不能由此推断所有成分级广度无效。

所有历史此前已被研究，机制又受复用差区间启发，因此即使通过也不能称新鲜样本外。当前没有新增完整策略收益或前向证据，生产SIM未改。下一步是SOP中的当时成分与权重、内部广度数据资格核查；不是将本候选上线。

[信息增量](information_increment.csv) · [训练成熟时点](training_audit.csv) · [研究SOP](../../docs/research_failure_to_feature_sop.md)

Recorder `d1240e470d8842119f1ea5c1b478b878`。
